In [9]:
import polars as pl

# =========================
# 🔧 FUNCIÓN LIMPIA Y GENÉRICA
# =========================

def cargar_y_normalizar(ruta):
    df = pl.read_csv(ruta)

    # detectar nombre de fecha
    if "date" in df.columns:
        fecha_col = "date"
    elif "Fecha" in df.columns:
        fecha_col = "Fecha"
    else:
        raise ValueError(f"No se encontró columna de fecha en {ruta}")

    # convertir a formato estándar
    df = df.with_columns(
        pl.col(fecha_col)
        .cast(pl.Utf8)
        .str.slice(0, 10)  # YYYY-MM-DD
        .str.strptime(pl.Date, format="%Y-%m-%d")
        .alias("date")
    )

    # eliminar columna original si no es "date"
    if fecha_col != "date":
        df = df.drop(fecha_col)

    return df


# =========================
# 📥 CARGAR TODO
# =========================

pvpc = cargar_y_normalizar("/Users/macbook/ProyectosLocales/PrecioLuz/datos/pvpc_total.csv")
gas = cargar_y_normalizar("/Users/macbook/ProyectosLocales/PrecioLuz/datos/gas_total.csv")
demanda = cargar_y_normalizar("/Users/macbook/ProyectosLocales/PrecioLuz/datos/demanda_total.csv")
renovables = cargar_y_normalizar("/Users/macbook/ProyectosLocales/PrecioLuz/datos/renovables_total.csv")

# =========================
# 🔗 MERGE
# =========================

df = pvpc.join(gas, on="date", how="outer", coalesce=True)
df = df.join(demanda, on="date", how="outer", coalesce=True)
df = df.join(renovables, on="date", how="outer", coalesce=True)

# =========================
# 🧹 ORDENAR
# =========================

df = df.sort("date")

# cambiar nombres de las columnas
df = df.rename({
    "date": "Fecha",
    "pvpc": "Luz",
    "Precio_gas": "Gas",
    "value": "Energia",
    "solar": "Solar",
    "eolica": "Eolica",
    "renovable_total": "Renovables"
})
# =========================
# 🔍 VALIDACIÓN
# =========================

print("\n--- INFO GENERAL ---")
print("Filas:", df.height)
print("Columnas:", df.columns)

print("\n--- PRIMERAS FILAS ---")
print(df.head(5))

print("\n--- RANGO FECHAS ---")
print(df["Fecha"].min(), df["Fecha"].max())

print("\n--- NULOS ---")
print(df.null_count())

print("\n--- DUPLICADOS ---")
print(df.height - df.unique(subset=["Fecha"]).height)


--- INFO GENERAL ---
Filas: 4018
Columnas: ['Fecha', 'Luz', 'Gas', 'Energia', 'Solar', 'Eolica', 'Renovables']

--- PRIMERAS FILAS ---
shape: (5, 7)
┌────────────┬──────────┬──────┬──────────────┬───────┬────────┬────────────┐
│ Fecha      ┆ Luz      ┆ Gas  ┆ Energia      ┆ Solar ┆ Eolica ┆ Renovables │
│ ---        ┆ ---      ┆ ---  ┆ ---          ┆ ---   ┆ ---    ┆ ---        │
│ date       ┆ f64      ┆ f64  ┆ f64          ┆ f64   ┆ f64    ┆ f64        │
╞════════════╪══════════╪══════╪══════════════╪═══════╪════════╪════════════╡
│ 2015-01-01 ┆ 0.082174 ┆ null ┆ 23891.201389 ┆ null  ┆ null   ┆ null       │
│ 2015-01-02 ┆ 0.087778 ┆ null ┆ 28260.5625   ┆ null  ┆ null   ┆ null       │
│ 2015-01-03 ┆ 0.083587 ┆ null ┆ 27586.979167 ┆ null  ┆ null   ┆ null       │
│ 2015-01-04 ┆ 0.072875 ┆ null ┆ 26235.395833 ┆ null  ┆ null   ┆ null       │
│ 2015-01-05 ┆ 0.092677 ┆ null ┆ 28227.111111 ┆ null  ┆ null   ┆ null       │
└────────────┴──────────┴──────┴──────────────┴───────┴────────┴──────

/var/folders/hx/0zmhrp2d37x4jg59pl9ktf0r0000gn/T/ipykernel_87392/2363912191.py:47: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df = pvpc.join(gas, on="date", how="outer", coalesce=True)
/var/folders/hx/0zmhrp2d37x4jg59pl9ktf0r0000gn/T/ipykernel_87392/2363912191.py:48: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df = df.join(demanda, on="date", how="outer", coalesce=True)
/var/folders/hx/0zmhrp2d37x4jg59pl9ktf0r0000gn/T/ipykernel_87392/2363912191.py:49: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df = df.join(renovables, on="date", how="outer", coalesce=True)


In [10]:
# =========================
# 💾 GUARDAR
# =========================

df.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/limpio_total.csv")